In [1]:
# ============================================================
# PS S6E5 - exp_20260504_010_blend_core_min
#
# Compare:
# - Avg
# - Ridge
# - ElasticNet - disabled
# - LogisticRegression
# - Hill Climb
# - Nelder-Mead
# - Signed SLSQP
# ============================================================

In [2]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from scipy.optimize import minimize
from scipy.stats import rankdata, spearmanr

from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, ElasticNet, LogisticRegression

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260504_010_blend_core_min"
    COMPETITION = "PS S6E5 Predicting F1 Pit Stops"
    METRIC = "AUC"

    COMP_BASE = Path("/kaggle/input/competitions/playground-series-s6e5")
    TRAIN_PATH = COMP_BASE / "train.csv"
    SUB_PATH = COMP_BASE / "sample_submission.csv"

    NPY_BASE = Path("/kaggle/input/datasets/mizushimatoshihiko/npy-files")

    OUTDIR = Path(f"/kaggle/working/{EXP_ID}")

    ID_COL = "id"
    TARGET = "PitNextLap"

    SEED = 42
    N_SPLITS = 5

    EPS = 1e-6


CFG.OUTDIR.mkdir(parents=True, exist_ok=True)

In [4]:
# ============================================================
# Utilities
# ============================================================

def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)


def safe_clip(x, eps=CFG.EPS):
    return np.clip(np.asarray(x, dtype=float), eps, 1.0 - eps)


def safe_logit(p, eps=CFG.EPS):
    p = safe_clip(p, eps)
    return np.log(p / (1.0 - p))


def rank01(x):
    x = np.asarray(x, dtype=float)
    return rankdata(x, method="average") / (len(x) + 1.0)


def softmax(z):
    z = np.asarray(z, dtype=float)
    z = z - np.max(z)
    e = np.exp(z)
    return e / e.sum()


def normalize_weights(w):
    w = np.asarray(w, dtype=float)
    w = np.clip(w, 0.0, None)
    s = w.sum()
    if s <= 0:
        return np.ones_like(w) / len(w)
    return w / s


def weighted_average(X, w):
    w = normalize_weights(w)
    return np.asarray(X) @ w


def auc(y, pred):
    return roc_auc_score(y, pred)


def sanitize_name(name: str) -> str:
    return (
        name.replace(" ", "_")
            .replace("/", "_")
            .replace("+", "plus")
            .replace("-", "_")
            .replace(".", "_")
    )


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load data
# ============================================================

train = pd.read_csv(CFG.TRAIN_PATH)
sub = pd.read_csv(CFG.SUB_PATH)

y = train[CFG.TARGET].astype(int).values

print("train:", train.shape)
print("sub  :", sub.shape)
print("target mean:", y.mean())

train: (439140, 16)
sub  : (188165, 2)
target mean: 0.19898210137996994


In [6]:
# ============================================================
# Load OOF / pred
# ============================================================

MODEL_SPECS = [
    {
        "key": "007",
        "name": "007_realmlp_shared001_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260503_007_realmlp_shared001_min.npy",
        "pred": "pred_exp_20260503_007_realmlp_shared001_min.npy",
        "public_lb": 0.95273,
    },
    # {
    #     "key": "009",
    #     "name": "009_lgb_shared003_full_fe_single",
    #     "family": "LightGBM",
    #     "oof": "oof_exp_20260503_009_lgb_shared003_full_fe_single_min.npy",
    #     "pred": "pred_exp_20260503_009_lgb_shared003_full_fe_single_min.npy",
    #     "public_lb": 0.95076,
    # },
    {
        "key": "014",
        "name": "cat_shared004_no_tyreageratio_str_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
        "pred": "pred_exp_20260508_014_cat_shared004_no_tyreageratio_str_min.npy",
        "public_lb": 0.95233,
    },
    {
        "key": "015",
        "name": "cat_shared004_no_race_year_groupstats_min",
        "family": "CatBoost",
        "oof": "oof_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
        "pred": "pred_exp_20260508_015_cat_shared004_no_race_year_groupstats_min.npy",
        "public_lb": 0.95244,
    },
    {
        "key": "016",
        "name": "xgb_shared005_fe_te_seedens_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "pred": "pred_exp_20260508_016_xgb_shared005_fe_te_seedens_min.npy",
        "public_lb": 0.95164,
    },
    {
        "key": "017",
        "name": "xgb_shared005_no_pitstop_pairte_min",
        "family": "XGBoost",
        "oof": "oof_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
        "pred": "pred_exp_20260509_017_xgb_shared005_no_pitstop_pairte_min.npy",
        "public_lb": 0.95164,
    },
    {
        "key": "018",
        "name": "realmlp_shared006_pipeline_a_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260510_018_realmlp_shared006_pipeline_a_min.npy",
        "pred": "pred_exp_20260510_018_realmlp_shared006_pipeline_a_min.npy",
        "public_lb": 0.95316,
    },
    # {
    #     "key": "020",
    #     "name": "lgb_progress_window_te_min",
    #     "family": "LightGBM",
    #     "oof": "oof_exp_20260511_020_lgb_progress_window_te_min.npy",
    #     "pred": "pred_exp_20260511_020_lgb_progress_window_te_min.npy",
    #     "public_lb": 0.95075,
    # },
    {
        "key": "021",
        "name": "tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min",
        "family": "TabM",
        "oof": "oof_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.npy",
        "pred": "pred_exp_20260511_021_tabm_shared007_comp_oof_no_pitstop_no_isoriginaldata_min.npy",
        "public_lb": 0.95171,
    },
    {
        "key": "022",
        "name": "realmlp_shared006_pipeline_a_no_race_year_te_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260512_022_realmlp_shared006_pipeline_a_no_race_year_te_min.npy",
        "pred": "pred_exp_20260512_022_realmlp_shared006_pipeline_a_no_race_year_te_min.npy",
        "public_lb": 0.95301,
    },
    {
        "key": "023",
        "name": "realmlp_shared006_pipeline_a_no_pitstop_cat_count_min",
        "family": "RealMLP",
        "oof": "oof_exp_20260512_023_realmlp_shared006_pipeline_a_no_pitstop_cat_count_min.npy",
        "pred": "pred_exp_20260512_023_realmlp_shared006_pipeline_a_no_pitstop_cat_count_min.npy",
        "public_lb": 0.95304,
    },
    {
        "key": "024",
        "name": "lgb_progress_window_weather_agg_min",
        "family": "LightGBM",
        "oof": "oof_exp_20260513_024_lgb_progress_window_weather_agg_min.npy",
        "pred": "pred_exp_20260513_024_lgb_progress_window_weather_agg_min.npy",
        "public_lb": 0.95086,
    },
    {
        "key": "025",
        "name": "lgb_year_stint_flags_min",
        "family": "LightGBM",
        "oof": "oof_exp_20260513_025_lgb_year_stint_flags_min.npy",
        "pred": "pred_exp_20260513_025_lgb_year_stint_flags_min.npy",
        "public_lb": 0.95092,
    },
]

oofs = {}
preds = {}

for spec in MODEL_SPECS:
    oof_path = CFG.NPY_BASE / spec["oof"]
    pred_path = CFG.NPY_BASE / spec["pred"]

    assert oof_path.exists(), f"missing oof: {oof_path}"
    assert pred_path.exists(), f"missing pred: {pred_path}"

    oof = np.load(oof_path)
    pred = np.load(pred_path)

    assert len(oof) == len(train), (spec["key"], len(oof), len(train))
    assert len(pred) == len(sub), (spec["key"], len(pred), len(sub))
    assert np.isfinite(oof).all(), spec["key"]
    assert np.isfinite(pred).all(), spec["key"]

    oofs[spec["key"]] = oof.astype(float)
    preds[spec["key"]] = pred.astype(float)

model_keys = [s["key"] for s in MODEL_SPECS]
model_names = [s["name"] for s in MODEL_SPECS]

X_raw = np.column_stack([oofs[k] for k in model_keys])
T_raw = np.column_stack([preds[k] for k in model_keys])

print("X_raw:", X_raw.shape)
print("T_raw:", T_raw.shape)

X_raw: (439140, 11)
T_raw: (188165, 11)


In [7]:
# ============================================================
# Individual reports
# ============================================================

individual_rows = []

for i, spec in enumerate(MODEL_SPECS):
    pred_oof = X_raw[:, i]
    individual_rows.append({
        "key": spec["key"],
        "name": spec["name"],
        "family": spec["family"],
        "cv_auc": auc(y, pred_oof),
        "public_lb": spec["public_lb"],
        "oof_min": float(pred_oof.min()),
        "oof_max": float(pred_oof.max()),
        "pred_min": float(T_raw[:, i].min()),
        "pred_max": float(T_raw[:, i].max()),
    })

individual_df = pd.DataFrame(individual_rows).sort_values("cv_auc", ascending=False)
display(individual_df)

best_single_key = individual_df.iloc[0]["key"]
best_single_auc = individual_df.iloc[0]["cv_auc"]

print("best single:", best_single_key, best_single_auc)

,key,name,family,cv_auc,public_lb,oof_min,oof_max,pred_min,pred_max
5,018,realmlp_shared006_pipeline_a_min,RealMLP,0.953476,0.95316,1.100145e-05,0.998269,0.000023,0.997298
8,023,realmlp_shared006_pipeline_a_no_pitstop_cat_co...,RealMLP,0.953446,0.95304,1.438200e-05,0.998677,0.000022,0.997378
7,022,realmlp_shared006_pipeline_a_no_race_year_te_min,RealMLP,0.953431,0.95301,1.483063e-05,0.998356,0.000021,0.997583
0,007,007_realmlp_shared001_min,RealMLP,0.953270,0.95273,1.565386e-05,0.998289,0.000027,0.997671
1,014,cat_shared004_no_tyreageratio_str_min,CatBoost,0.952726,0.95233,7.059056e-05,0.997215,0.000153,0.996355
2,015,cat_shared004_no_race_year_groupstats_min,CatBoost,0.952714,0.95244,4.489366e-05,0.997058,0.000141,0.996636
3,016,xgb_shared005_fe_te_seedens_min,XGBoost,0.951986,0.95164,1.674493e-05,0.997698,0.000027,0.997298
4,017,xgb_shared005_no_pitstop_pairte_min,XGBoost,0.951939,0.95164,1.331864e-05,0.997702,0.000024,0.997621
6,021,tabm_shared007_comp_oof_no_pitstop_no_isorigin...,TabM,0.951492,0.95171,1.000000e-06,0.999431,0.000003,0.996751
9,024,lgb_progress_window_weather_agg_min,LightGBM,0.951178,0.95086,1.668872e-05,0.999032,0.000020,0.998800


best single: 018 0.953475993350039


In [8]:
# ============================================================
# Correlation matrix
# ============================================================

pearson = pd.DataFrame(
    np.corrcoef(X_raw.T),
    index=model_keys,
    columns=model_keys,
)

spearman_mat = np.zeros((len(model_keys), len(model_keys)))

for i in range(len(model_keys)):
    for j in range(len(model_keys)):
        spearman_mat[i, j] = spearmanr(X_raw[:, i], X_raw[:, j]).correlation

spearman_df = pd.DataFrame(
    spearman_mat,
    index=model_keys,
    columns=model_keys,
)

print("Pearson correlation")
display(pearson)

print("Spearman correlation")
display(spearman_df)

pearson.to_csv(CFG.OUTDIR / "corr_pearson.csv")
spearman_df.to_csv(CFG.OUTDIR / "corr_spearman.csv")

Pearson correlation


,007,014,015,016,017,018,021,022,023,024,025
007,1.000000,0.962718,0.961711,0.981305,0.981032,0.995835,0.982198,0.995756,0.995827,0.958067,0.957319
014,0.962718,1.000000,0.996917,0.962898,0.962739,0.962634,0.968765,0.962570,0.962603,0.984981,0.984727
015,0.961711,0.996917,1.000000,0.962398,0.962237,0.961623,0.967914,0.961611,0.961621,0.984974,0.985005
016,0.981305,0.962898,0.962398,1.000000,0.998485,0.981123,0.978284,0.981152,0.981087,0.960092,0.960231
017,0.981032,0.962739,0.962237,0.998485,1.000000,0.980833,0.978150,0.980840,0.980827,0.960679,0.960841
018,0.995835,0.962634,0.961623,0.981123,0.980833,1.000000,0.982275,0.996976,0.997029,0.957949,0.957175
021,0.982198,0.968765,0.967914,0.978284,0.978150,0.982275,1.000000,0.982297,0.982287,0.966188,0.965632
022,0.995756,0.962570,0.961611,0.981152,0.980840,0.996976,0.982297,1.000000,0.998286,0.957887,0.957171
023,0.995827,0.962603,0.961621,0.981087,0.980827,0.997029,0.982287,0.998286,1.000000,0.958000,0.957208
024,0.958067,0.984981,0.984974,0.960092,0.960679,0.957949,0.966188,0.957887,0.958000,1.000000,0.997920


Spearman correlation


,007,014,015,016,017,018,021,022,023,024,025
007,1.000000,0.976924,0.976733,0.974170,0.973720,0.993611,0.970593,0.993601,0.993648,0.966711,0.967143
014,0.976924,1.000000,0.994676,0.975612,0.975146,0.976431,0.970041,0.976682,0.976901,0.973130,0.973214
015,0.976733,0.994676,1.000000,0.975249,0.974943,0.976373,0.969630,0.976505,0.976754,0.972737,0.973030
016,0.974170,0.975612,0.975249,1.000000,0.998203,0.973029,0.965983,0.973239,0.973342,0.969182,0.970269
017,0.973720,0.975146,0.974943,0.998203,1.000000,0.972656,0.965901,0.972833,0.972973,0.969729,0.970857
018,0.993611,0.976431,0.976373,0.973029,0.972656,1.000000,0.970461,0.995831,0.995901,0.966222,0.966597
021,0.970593,0.970041,0.969630,0.965983,0.965901,0.970461,1.000000,0.970631,0.970775,0.962801,0.963410
022,0.993601,0.976682,0.976505,0.973239,0.972833,0.995831,0.970631,1.000000,0.997677,0.966277,0.966711
023,0.993648,0.976901,0.976754,0.973342,0.972973,0.995901,0.970775,0.997677,1.000000,0.966593,0.966931
024,0.966711,0.973130,0.972737,0.969182,0.969729,0.966222,0.962801,0.966277,0.966593,1.000000,0.996733


In [9]:
# ============================================================
# Meta feature builders
# ============================================================

def build_meta_features(X, T):
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)

    X_rank = np.column_stack([rank01(X[:, i]) for i in range(X.shape[1])])
    T_rank = np.column_stack([rank01(T[:, i]) for i in range(T.shape[1])])

    X_logit = np.column_stack([safe_logit(X[:, i]) for i in range(X.shape[1])])
    T_logit = np.column_stack([safe_logit(T[:, i]) for i in range(T.shape[1])])

    X_meta = np.column_stack([X, X_rank, X_logit])
    T_meta = np.column_stack([T, T_rank, T_logit])

    feature_names = (
        [f"raw_{k}" for k in model_keys] +
        [f"rank_{k}" for k in model_keys] +
        [f"logit_{k}" for k in model_keys]
    )

    return X_meta, T_meta, feature_names


X_meta, T_meta, meta_feature_names = build_meta_features(X_raw, T_raw)

print("X_meta:", X_meta.shape)
print("T_meta:", T_meta.shape)
print("meta features:", meta_feature_names)

X_meta: (439140, 33)
T_meta: (188165, 33)
meta features: ['raw_007', 'raw_014', 'raw_015', 'raw_016', 'raw_017', 'raw_018', 'raw_021', 'raw_022', 'raw_023', 'raw_024', 'raw_025', 'rank_007', 'rank_014', 'rank_015', 'rank_016', 'rank_017', 'rank_018', 'rank_021', 'rank_022', 'rank_023', 'rank_024', 'rank_025', 'logit_007', 'logit_014', 'logit_015', 'logit_016', 'logit_017', 'logit_018', 'logit_021', 'logit_022', 'logit_023', 'logit_024', 'logit_025']


In [10]:
# ============================================================
# Save candidate helper
# ============================================================

candidate_records = {}
candidate_summary = []

def add_candidate(
    name,
    method,
    oof_pred,
    test_pred,
    weights=None,
    params=None,
    notes=None,
):
    oof_pred = np.asarray(oof_pred, dtype=float)
    test_pred = np.asarray(test_pred, dtype=float)

    score = auc(y, oof_pred)

    candidate_records[name] = {
        "method": method,
        "oof": oof_pred,
        "pred": test_pred,
        "score": float(score),
        "weights": None if weights is None else [float(x) for x in weights],
        "params": params or {},
        "notes": notes or "",
    }

    candidate_summary.append({
        "name": name,
        "method": method,
        "cv_auc": float(score),
        f"delta_vs_{best_single_key}": float(score - auc(y, oofs[best_single_key])),
        "delta_vs_best_single": float(score - best_single_auc),
        "weights": None if weights is None else json.dumps([round(float(x), 8) for x in weights]),
        "params": json.dumps(params or {}, ensure_ascii=False),
        "notes": notes or "",
        "oof_min": float(oof_pred.min()),
        "oof_max": float(oof_pred.max()),
        "pred_min": float(test_pred.min()),
        "pred_max": float(test_pred.max()),
    })

    print(f"{name}: {score:.9f}")

In [11]:
# ============================================================
# Simple averages
# ============================================================

print("\n" + "=" * 80)
print("Simple averages")
print("=" * 80)

avg_specs = {
    # "avg_007_014_015_016_018_020_021_022_023": ["007", "014", "015", "016", "018", "020", "021", "022", "023"],
    # "avg_007_014_016_018_020_021_022_023": ["007", "014", "016", "018", "020", "021", "022", "023"],
    # "avg_007_014_015_018_020_021_022_023": ["007", "014", "015", "018", "020", "021", "022", "023"],
    "avg" : model_keys
}

for name, keys in avg_specs.items():
    idx = [model_keys.index(k) for k in keys]
    w = np.zeros(len(model_keys))
    for j in idx:
        w[j] = 1.0 / len(idx)

    add_candidate(
        name=name,
        method="avg",
        oof_pred=weighted_average(X_raw, w),
        test_pred=weighted_average(T_raw, w),
        weights=w,
        notes=f"simple average of {keys}",
    )


Simple averages
avg: 0.954351930


In [12]:
# ============================================================
# Ridge / ElasticNet / LogisticRegression meta CV
# Two-stage search
# ============================================================

meta_folds = list(
    StratifiedKFold(
        n_splits=CFG.N_SPLITS,
        shuffle=True,
        random_state=CFG.SEED,
    ).split(X_meta, y)
)


def make_refined_grid_1d(best_value, factor_low=0.25, factor_high=4.0, n=17, min_value=1e-8):
    lo = max(best_value * factor_low, min_value)
    hi = max(best_value * factor_high, lo * 1.01)
    return np.geomspace(lo, hi, n).tolist()


def run_meta_regressor_cv(estimator_factory, params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            estimator_factory(params),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict(X_meta[va_idx])

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_regressor_two_stage(name, estimator_factory, coarse_grid, refine_builder):
    history = []

    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_regressor_cv(estimator_factory, params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        estimator_factory(best["params"]),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict(T_meta)

    add_candidate(
        name=name,
        method=name.split("_")[0],
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


def run_meta_logreg_cv(params):
    oof_meta = np.zeros(len(y), dtype=float)

    for tr_idx, va_idx in meta_folds:
        model = make_pipeline(
            StandardScaler(),
            LogisticRegression(
                C=params["C"],
                penalty="l2",
                solver="lbfgs",
                max_iter=3000,
                random_state=CFG.SEED,
            ),
        )
        model.fit(X_meta[tr_idx], y[tr_idx])
        oof_meta[va_idx] = model.predict_proba(X_meta[va_idx])[:, 1]

    score = auc(y, oof_meta)
    return score, oof_meta


def run_meta_logreg_two_stage(name, coarse_grid, refine_builder):
    history = []
    best = None

    print(f"\n{name} coarse search")
    for params in coarse_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "coarse", "score": float(score), **params})
        print(params, score)

        if best is None or score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    refined_grid = refine_builder(best["params"])

    print(f"\n{name} refined search around {best['params']}")
    for params in refined_grid:
        score, oof_meta = run_meta_logreg_cv(params)
        history.append({"stage": "refined", "score": float(score), **params})
        print(params, score)

        if score > best["score"]:
            best = {
                "score": score,
                "params": params,
                "oof": oof_meta,
            }

    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=best["params"]["C"],
            penalty="l2",
            solver="lbfgs",
            max_iter=3000,
            random_state=CFG.SEED,
        ),
    )
    final_model.fit(X_meta, y)
    test_meta = final_model.predict_proba(T_meta)[:, 1]

    add_candidate(
        name=name,
        method="logreg",
        oof_pred=best["oof"],
        test_pred=test_meta,
        params=best["params"],
        notes="two-stage meta CV logistic regression on raw+rank+logit features",
    )

    hist_df = pd.DataFrame(history).sort_values("score", ascending=False)
    hist_df.to_csv(CFG.OUTDIR / f"{name}_search_history.csv", index=False)

    print(f"\n{name} best:", best["params"], best["score"])
    display(hist_df.head(30))

    return best, hist_df


print("\n" + "=" * 80)
print("Ridge / ElasticNet / LogReg two-stage search")
print("=" * 80)


# ----------------------------
# Ridge two-stage
# ----------------------------

ridge_coarse_grid = [
    {"alpha": a}
    for a in np.geomspace(1e-3, 1e3, 19)
]

ridge_best, ridge_hist = run_meta_regressor_two_stage(
    name="ridge_meta_all",
    estimator_factory=lambda p: Ridge(
        alpha=p["alpha"],
        random_state=CFG.SEED,
    ),
    coarse_grid=ridge_coarse_grid,
    refine_builder=lambda best_p: [
        {"alpha": a}
        for a in make_refined_grid_1d(
            best_p["alpha"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


# ----------------------------
# ElasticNet two-stage
# ----------------------------

elastic_coarse_grid = [
    {"alpha": a, "l1_ratio": l1}
    for a in np.geomspace(1e-4, 1e-1, 7)
    for l1 in [0.05, 0.10, 0.20, 0.50, 0.90]
]


def build_elastic_refined_grid_fast(best_p):
    alpha_grid = make_refined_grid_1d(
        best_p["alpha"],
        factor_low=0.5,
        factor_high=2.0,
        n=7,
        min_value=1e-7,
    )

    l1_center = best_p["l1_ratio"]
    l1_grid = sorted(set([
        round(max(0.001, l1_center - 0.10), 6),
        round(l1_center, 6),
        round(min(0.999, l1_center + 0.10), 6),
        0.05,
        0.10,
        0.20,
        0.50,
        0.90,
    ]))

    return [
        {"alpha": a, "l1_ratio": l1}
        for a in alpha_grid
        for l1 in l1_grid
    ]


# elastic_best, elastic_hist = run_meta_regressor_two_stage(
#     name="elasticnet_meta_all",
#     estimator_factory=lambda p: ElasticNet(
#         alpha=p["alpha"],
#         l1_ratio=p["l1_ratio"],
#         max_iter=30000,
#         random_state=CFG.SEED,
#         selection="cyclic",
#     ),
#     coarse_grid=elastic_coarse_grid,
#     refine_builder=build_elastic_refined_grid_fast,
# )


# ----------------------------
# LogisticRegression two-stage
# ----------------------------

logreg_coarse_grid = [
    {"C": c}
    for c in np.geomspace(1e-3, 1e3, 19)
]

logreg_best, logreg_hist = run_meta_logreg_two_stage(
    name="logreg_meta_all",
    coarse_grid=logreg_coarse_grid,
    refine_builder=lambda best_p: [
        {"C": c}
        for c in make_refined_grid_1d(
            best_p["C"],
            factor_low=0.2,
            factor_high=5.0,
            n=21,
            min_value=1e-6,
        )
    ],
)


Ridge / ElasticNet / LogReg two-stage search

ridge_meta_all coarse search
{'alpha': np.float64(0.001)} 0.9538712676279356
{'alpha': np.float64(0.0021544346900318843)} 0.9538712685063543
{'alpha': np.float64(0.004641588833612777)} 0.95387127081627
{'alpha': np.float64(0.01)} 0.9538712755662373
{'alpha': np.float64(0.021544346900318832)} 0.9538712846757637
{'alpha': np.float64(0.046415888336127774)} 0.9538713093365534
{'alpha': np.float64(0.1)} 0.9538713435948797
{'alpha': np.float64(0.21544346900318823)} 0.95387143085113
{'alpha': np.float64(0.46415888336127775)} 0.9538716003533905
{'alpha': np.float64(1.0)} 0.9538719476540878
{'alpha': np.float64(2.154434690031882)} 0.9538726176272267
{'alpha': np.float64(4.6415888336127775)} 0.9538738411017017
{'alpha': np.float64(10.0)} 0.9538755717165295
{'alpha': np.float64(21.54434690031882)} 0.953877238402002
{'alpha': np.float64(46.41588833612773)} 0.9538769819037635
{'alpha': np.float64(100.0)} 0.9538713442130261
{'alpha': np.float64(215.4434

,stage,score,alpha
31,refined,0.953878,29.725374
30,refined,0.953877,25.306398
32,refined,0.953877,34.915988
33,refined,0.953877,41.012981
29,refined,0.953877,21.544347
13,coarse,0.953877,21.544347
28,refined,0.953877,18.341563
14,coarse,0.953877,46.415888
34,refined,0.953877,48.174624
27,refined,0.953877,15.614905



logreg_meta_all coarse search
{'C': np.float64(0.001)} 0.9544002983683031
{'C': np.float64(0.0021544346900318843)} 0.9544033589739979
{'C': np.float64(0.004641588833612777)} 0.9544045333546318
{'C': np.float64(0.01)} 0.9544030299574217
{'C': np.float64(0.021544346900318832)} 0.954401784294743
{'C': np.float64(0.046415888336127774)} 0.9544007035795615
{'C': np.float64(0.1)} 0.9544000397878871
{'C': np.float64(0.21544346900318823)} 0.9544005599743832
{'C': np.float64(0.46415888336127775)} 0.9544008987511665
{'C': np.float64(1.0)} 0.9544010094969814
{'C': np.float64(2.154434690031882)} 0.9544010512706671
{'C': np.float64(4.6415888336127775)} 0.9544004514408821
{'C': np.float64(10.0)} 0.9544004434375124
{'C': np.float64(21.54434690031882)} 0.9544004396635659
{'C': np.float64(46.41588833612773)} 0.9544004357269489
{'C': np.float64(100.0)} 0.9544004343930539
{'C': np.float64(215.44346900318823)} 0.954400434620792
{'C': np.float64(464.1588833612773)} 0.9544004336447715
{'C': np.float64(1000.

,stage,score,C
27,refined,0.954406,0.003364
30,refined,0.954405,0.005452
28,refined,0.954405,0.003952
2,coarse,0.954405,0.004642
29,refined,0.954405,0.004642
26,refined,0.954404,0.002864
32,refined,0.954404,0.007522
23,refined,0.954404,0.001767
25,refined,0.954404,0.002438
31,refined,0.954404,0.006404


In [13]:
# ============================================================
# Hill Climb non-negative weights
# ============================================================

print("\n" + "=" * 80)
print("Hill Climb")
print("=" * 80)

def hill_climb_weights(X, y, init_candidates=None):
    n = X.shape[1]

    candidates = []

    # one-hot starts
    for i in range(n):
        w = np.zeros(n)
        w[i] = 1.0
        candidates.append(w)

    # avg start
    candidates.append(np.ones(n) / n)

    if init_candidates:
        for w in init_candidates:
            candidates.append(normalize_weights(w))

    best_w = None
    best_score = -np.inf

    for w in candidates:
        s = auc(y, weighted_average(X, w))
        if s > best_score:
            best_score = s
            best_w = normalize_weights(w)

    for step in [0.20, 0.10, 0.05, 0.02, 0.01, 0.005, 0.002, 0.001]:
        improved = True

        while improved:
            improved = False

            for i in range(n):
                for direction in [-1, 1]:
                    trial = best_w.copy()
                    trial[i] += direction * step
                    trial = normalize_weights(trial)

                    s = auc(y, weighted_average(X, trial))

                    if s > best_score + 1e-12:
                        best_score = s
                        best_w = trial
                        improved = True

    return best_w, best_score


hc_init = []

# Use candidates that already looked good as starts
for cand_name in ["avg_007_008_009", "avg_006b_007_008_009"]:
    if cand_name in candidate_records:
        hc_init.append(candidate_records[cand_name]["weights"])

hc_w, hc_score = hill_climb_weights(X_raw, y, init_candidates=hc_init)

add_candidate(
    name="hc_nonnegative_raw",
    method="hc",
    oof_pred=weighted_average(X_raw, hc_w),
    test_pred=weighted_average(T_raw, hc_w),
    weights=hc_w,
    params={"constraint": "nonnegative_sum1"},
    notes="greedy hill climb on raw OOF predictions; in-sample optimizer, interpret cautiously",
)

print("HC weights:")
for k, w in zip(model_keys, hc_w):
    print(k, w)


Hill Climb
hc_nonnegative_raw: 0.954429744
HC weights:
007 0.10636541617004905
014 0.10733527314222895
015 0.07862111462916771
016 0.14963772792481553
017 0.005837077189321283
018 0.19689028540868322
021 0.0757135746991786
022 0.1012048618441413
023 0.09487163315082876
024 0.034343194186898116
025 0.0491798416546875


In [14]:
# ============================================================
# Nelder-Mead softmax weights
# ============================================================

print("\n" + "=" * 80)
print("Nelder-Mead")
print("=" * 80)

def nm_optimize_weights(X, y, start_w=None, maxiter=500):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n

    start_w = normalize_weights(start_w)
    z0 = np.log(np.clip(start_w, 1e-8, 1.0))

    def objective(z):
        w = softmax(z)
        p = weighted_average(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        z0,
        method="Nelder-Mead",
        options={
            "maxiter": maxiter,
            "xatol": 1e-7,
            "fatol": 1e-10,
            "disp": True,
        },
    )

    w = softmax(res.x)
    score = auc(y, weighted_average(X, w))

    return w, score, res


nm_w, nm_score, nm_res = nm_optimize_weights(
    X_raw,
    y,
    start_w=hc_w,
    maxiter=500,
)

add_candidate(
    name="nm_softmax_raw",
    method="nm",
    oof_pred=weighted_average(X_raw, nm_w),
    test_pred=weighted_average(T_raw, nm_w),
    weights=nm_w,
    params={
        "constraint": "softmax_sum1",
        "success": bool(nm_res.success),
        "message": str(nm_res.message),
        "nit": int(getattr(nm_res, "nit", -1)),
        "fun": float(nm_res.fun),
    },
    notes="Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously",
)

print("NM weights:")
for k, w in zip(model_keys, nm_w):
    print(k, w)


Nelder-Mead
Optimization terminated successfully.
         Current function value: -0.954430
         Iterations: 235
         Function evaluations: 522
nm_softmax_raw: 0.954429784
NM weights:
007 0.10670498794694636
014 0.10723326908125838
015 0.07742585459259926
016 0.1498689122132076
017 0.005500802285130832
018 0.19735107103024876
021 0.07526569407586517
022 0.10108449158806807
023 0.09598374531699083
024 0.03442140298180323
025 0.049159768887881435


In [15]:
# ============================================================
# Signed SLSQP weights
# Allow small negative weights, with sum(w)=1
# ============================================================

def weighted_average_signed(X, w):
    return np.asarray(X, dtype=float) @ np.asarray(w, dtype=float)


def optimize_signed_slsqp(
    X,
    y,
    start_w=None,
    neg_bound=-0.10,
    pos_bound=0.60,
    maxiter=1000,
):
    n = X.shape[1]

    if start_w is None:
        start_w = np.ones(n) / n
    else:
        start_w = np.asarray(start_w, dtype=float)

    # Start must satisfy sum=1
    start_w = start_w / start_w.sum()

    bounds = [(neg_bound, pos_bound) for _ in range(n)]

    constraints = [
        {
            "type": "eq",
            "fun": lambda w: np.sum(w) - 1.0,
        }
    ]

    def objective(w):
        p = weighted_average_signed(X, w)
        return -auc(y, p)

    res = minimize(
        objective,
        start_w,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={
            "maxiter": maxiter,
            "ftol": 1e-12,
            "disp": True,
        },
    )

    w = res.x
    score = auc(y, weighted_average_signed(X, w))

    return w, score, res

In [16]:
print("\n" + "=" * 80)
print("Signed SLSQP")
print("=" * 80)

signed_w, signed_score, signed_res = optimize_signed_slsqp(
    X_raw,
    y,
    start_w=nm_w,
    neg_bound=-0.05,
    pos_bound=0.50,
    maxiter=1000,
)

add_candidate(
    name="slsqp_signed_raw_neg005",
    method="slsqp_signed",
    oof_pred=weighted_average_signed(X_raw, signed_w),
    test_pred=weighted_average_signed(T_raw, signed_w),
    weights=signed_w,
    params={
        "constraint": "sum1_signed",
        "neg_bound": -0.05,
        "pos_bound": 0.50,
        "success": bool(signed_res.success),
        "message": str(signed_res.message),
        "nit": int(getattr(signed_res, "nit", -1)),
        "fun": float(signed_res.fun),
    },
    notes="signed SLSQP on raw OOF predictions; high overfit risk; attack-only",
)

print("Signed SLSQP weights:")
for k, w in zip(model_keys, signed_w):
    print(k, w)

print("Signed score:", signed_score)
print("sum weights:", signed_w.sum())
print("min weight:", signed_w.min())
print("max weight:", signed_w.max())


Signed SLSQP
Optimization terminated successfully    (Exit mode 0)
            Current function value: -0.9544297365839555
            Iterations: 12
            Function evaluations: 265
            Gradient evaluations: 12
slsqp_signed_raw_neg005: 0.954429737
Signed SLSQP weights:
007 0.10651097003393116
014 0.10735166751689354
015 0.07774940486030452
016 0.14987221010768195
017 0.005810330110779774
018 0.19764387319763527
021 0.07510157702888137
022 0.10099225846792792
023 0.0957739654872936
024 0.0341775495064599
025 0.04901619368221114
Signed score: 0.9544297365839555
sum weights: 1.0000000000000002
min weight: 0.005810330110779774
max weight: 0.19764387319763527


In [17]:
# ============================================================
# Summary
# ============================================================

summary_df = pd.DataFrame(candidate_summary).sort_values("cv_auc", ascending=False)
display(summary_df)

summary_df.to_csv(CFG.OUTDIR / "blend_summary.csv", index=False)

# Expand weights table
weights_rows = []

for name, rec in candidate_records.items():
    if rec["weights"] is None:
        continue

    row = {
        "candidate": name,
        "method": rec["method"],
        "cv_auc": rec["score"],
    }

    for k, w in zip(model_keys, rec["weights"]):
        row[f"w_{k}"] = float(w)

    weights_rows.append(row)

weights_df = pd.DataFrame(weights_rows).sort_values("cv_auc", ascending=False)
display(weights_df)

weights_df.to_csv(CFG.OUTDIR / "blend_weights.csv", index=False)

,name,method,cv_auc,delta_vs_018,delta_vs_best_single,weights,params,notes,oof_min,oof_max,pred_min,pred_max
4,nm_softmax_raw,nm,0.954430,0.000954,0.000954,"[0.10670499, 0.10723327, 0.07742585, 0.1498689...","{""constraint"": ""softmax_sum1"", ""success"": true...",Nelder-Mead on softmax weights; in-sample opti...,0.000055,0.996154,0.000056,0.996602
3,hc_nonnegative_raw,hc,0.954430,0.000954,0.000954,"[0.10636542, 0.10733527, 0.07862111, 0.1496377...","{""constraint"": ""nonnegative_sum1""}",greedy hill climb on raw OOF predictions; in-s...,0.000055,0.996154,0.000056,0.996597
5,slsqp_signed_raw_neg005,slsqp_signed,0.954430,0.000954,0.000954,"[0.10651097, 0.10735167, 0.0777494, 0.14987221...","{""constraint"": ""sum1_signed"", ""neg_bound"": -0....",signed SLSQP on raw OOF predictions; high over...,0.000055,0.996153,0.000056,0.996600
2,logreg_meta_all,logreg,0.954406,0.000930,0.000930,None,"{""C"": 0.003364129193756015}",two-stage meta CV logistic regression on raw+r...,0.000041,0.987059,0.000040,0.988595
0,avg,avg,0.954352,0.000876,0.000876,"[0.09090909, 0.09090909, 0.09090909, 0.0909090...",{},"simple average of ['007', '014', '015', '016',...",0.000064,0.996428,0.000058,0.996729
1,ridge_meta_all,ridge,0.953878,0.000402,0.000402,None,"{""alpha"": 29.725374455179868}",two-stage meta CV on raw+rank+logit features,-0.087193,1.012493,-0.071223,1.000844


,candidate,method,cv_auc,w_007,w_014,w_015,w_016,w_017,w_018,w_021,w_022,w_023,w_024,w_025
2,nm_softmax_raw,nm,0.954430,0.106705,0.107233,0.077426,0.149869,0.005501,0.197351,0.075266,0.101084,0.095984,0.034421,0.049160
1,hc_nonnegative_raw,hc,0.954430,0.106365,0.107335,0.078621,0.149638,0.005837,0.196890,0.075714,0.101205,0.094872,0.034343,0.049180
3,slsqp_signed_raw_neg005,slsqp_signed,0.954430,0.106511,0.107352,0.077749,0.149872,0.005810,0.197644,0.075102,0.100992,0.095774,0.034178,0.049016
0,avg,avg,0.954352,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909,0.090909


In [18]:
# ============================================================
# Save OOF / pred / submissions
# ============================================================

print("\n" + "=" * 80)
print("Save blend artifacts")
print("=" * 80)

submission_dir = CFG.OUTDIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

target_col = [c for c in sub.columns if c != CFG.ID_COL][0]

for name, rec in candidate_records.items():
    safe_name = sanitize_name(name)

    oof_path = CFG.OUTDIR / f"oof_{CFG.EXP_ID}_{safe_name}.npy"
    pred_path = CFG.OUTDIR / f"pred_{CFG.EXP_ID}_{safe_name}.npy"
    sub_path = submission_dir / f"sub_{safe_name}_{CFG.EXP_ID}.csv"

    np.save(oof_path, rec["oof"])
    np.save(pred_path, rec["pred"])

    sub_out = sub.copy()
    sub_out[target_col] = safe_clip(rec["pred"])
    sub_out.to_csv(sub_path, index=False)

    print(name, rec["score"], sub_path)


Save blend artifacts
avg 0.954351930084872 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_avg_exp_20260504_010_blend_core_min.csv
ridge_meta_all 0.9538775125987493 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_ridge_meta_all_exp_20260504_010_blend_core_min.csv
logreg_meta_all 0.9544056506705894 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_logreg_meta_all_exp_20260504_010_blend_core_min.csv
hc_nonnegative_raw 0.9544297444897232 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_hc_nonnegative_raw_exp_20260504_010_blend_core_min.csv
nm_softmax_raw 0.9544297835630848 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_nm_softmax_raw_exp_20260504_010_blend_core_min.csv
slsqp_signed_raw_neg005 0.9544297365839555 /kaggle/working/exp_20260504_010_blend_core_min/submissions/sub_slsqp_signed_raw_neg005_exp_20260504_010_blend_core_min.csv


In [19]:
# ============================================================
# Save result json
# ============================================================

result = {
    "experiment": {
        "id": CFG.EXP_ID,
        "competition": CFG.COMPETITION,
        "metric": CFG.METRIC,
        "created_at": "2026-05-04",
    },
    "inputs": {
        "model_specs": MODEL_SPECS,
        "model_keys": model_keys,
        "n_models": len(model_keys),
    },
    "individual": individual_df.to_dict(orient="records"),
    "correlation": {
        "pearson": pearson.to_dict(),
        "spearman": spearman_df.to_dict(),
    },
    "blend_summary": summary_df.to_dict(orient="records"),
    "blend_weights": weights_df.to_dict(orient="records"),
    "best": {
        "candidate": str(summary_df.iloc[0]["name"]),
        "method": str(summary_df.iloc[0]["method"]),
        "cv_auc": float(summary_df.iloc[0]["cv_auc"]),
    },
    "notes": [
        "This blend stage is intended to check whether 009 receives useful weight.",
        "Ridge/ElasticNet/LogReg use meta-CV on raw+rank+logit features.",
        "HC/NM optimize directly on full OOF and are more overfit-prone.",
        "If 009 receives meaningful weight, trying 009 + safe orig prior as next experiment is justified.",
        "If 009 receives little or zero weight, 009 + orig prior priority should be lowered.",
    ],
    "artifacts": {
        "outdir": str(CFG.OUTDIR),
        "blend_summary": str(CFG.OUTDIR / "blend_summary.csv"),
        "blend_weights": str(CFG.OUTDIR / "blend_weights.csv"),
        "submissions_dir": str(submission_dir),
    },
}

with open(CFG.OUTDIR / "blend_result.json", "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=2)

print("\nSaved to:", CFG.OUTDIR)
print("Best candidate:")
print(summary_df.iloc[0].to_dict())


Saved to: /kaggle/working/exp_20260504_010_blend_core_min
Best candidate:
{'name': 'nm_softmax_raw', 'method': 'nm', 'cv_auc': 0.9544297835630848, 'delta_vs_018': 0.0009537902130458686, 'delta_vs_best_single': 0.0009537902130458686, 'weights': '[0.10670499, 0.10723327, 0.07742585, 0.14986891, 0.0055008, 0.19735107, 0.07526569, 0.10108449, 0.09598375, 0.0344214, 0.04915977]', 'params': '{"constraint": "softmax_sum1", "success": true, "message": "Optimization terminated successfully.", "nit": 235, "fun": -0.9544297835630848}', 'notes': 'Nelder-Mead on softmax weights; in-sample optimizer, interpret cautiously', 'oof_min': 5.532468769808397e-05, 'oof_max': 0.9961538056806172, 'pred_min': 5.561849398170056e-05, 'pred_max': 0.9966024435615319}
